# DataPulse — Inference Walkthrough

Natural-language question → Gemini agent → `run_cypher` tool → Neo4j → answer.

### Prerequisites

- `.env` populated with `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, `GOOGLE_API_KEY` (see `.env.sample`)
- `data/raw/sales_1k.csv` exists — `uv run python -m src.datagen.generate`
- Graph loaded into Aura — `uv run python -m src.graph.builder.neo4j_graph_builder --csv data/raw/sales_1k.csv`

In [ ]:
import sys
import uuid
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.shared.config import Settings

settings = Settings()
assert settings.neo4j_uri, "NEO4J_URI not set in .env"
assert settings.google_api_key, "GOOGLE_API_KEY not set in .env"
print(f"neo4j:  {settings.neo4j_uri}")
print(f"model:  {settings.gemini_model}")

## 1. Open the graph and build the agent

One `Agent` with one tool (`run_cypher`) and a generated schema card as its system instruction.

In [ ]:
from src.graph.store.neo4j_store import Neo4jStore
from src.query_engine.agent.adk_agent import ask_async, build_agent

store = Neo4jStore.from_settings(settings)
agent = build_agent(store)

print(f"agent: {agent.name}  model: {agent.model}")
print(f"tools: {[t.__name__ for t in agent.tools]}")

## 2. Ask a question

The agent plans the Cypher, calls `run_cypher`, reads the rows, then writes the natural-language answer.

In [ ]:
question = "What are the top 3 product categories by total quantity sold?"
answer = await ask_async(agent, question)
print("Q:", question)
print("A:", answer)

## 3. Inspect the inference trace

Same flow as `ask_async`, but yields every event so you can see the Cypher the agent generated and the rows it received back.

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types


async def ask_with_trace(agent, question: str) -> tuple[str, list[tuple]]:
    runner = Runner(app_name=agent.name, agent=agent, session_service=InMemorySessionService())
    user_id, session_id = "demo", f"s-{uuid.uuid4().hex[:8]}"
    await runner.session_service.create_session(app_name=agent.name, user_id=user_id, session_id=session_id)
    msg = genai_types.Content(role="user", parts=[genai_types.Part(text=question)])
    trace: list[tuple] = []
    final = ""
    async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=msg):
        for part in (event.content.parts if event.content and event.content.parts else []):
            if getattr(part, "function_call", None):
                trace.append(("call", part.function_call.name, dict(part.function_call.args or {})))
            elif getattr(part, "function_response", None):
                trace.append(("result", part.function_response.name, part.function_response.response))
            elif getattr(part, "text", None) and event.is_final_response():
                final = part.text
    return final, trace


q = "Which region has the highest average unit price across all orders?"
answer, trace = await ask_with_trace(agent, q)

print(f"Q: {q}\n")
for kind, name, payload in trace:
    if kind == "call":
        print(f"-> {name}({payload.get('query', '')})")
    else:
        rows = payload.get("rows", []) if isinstance(payload, dict) else payload
        print(f"<- {name}: {len(rows) if isinstance(rows, list) else 'n/a'} row(s)")
        if isinstance(rows, list) and rows:
            print(f"   first row: {rows[0]}")
print(f"\nA: {answer}")

In [ ]:
store.close()